In [18]:
url_image_1 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/5uo16pKhdB1f2Vz7H8Utkg/image-1.png'
url_image_2 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fsuegY1q_OxKIxNhf6zeYg/image-2.png'
url_image_3 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/KCh_pM9BVWq_ZdzIBIA9Fw/image-3.png'
url_image_4 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/VaaYLw52RaykwrE3jpFv7g/image-4.png'
url_image_5 = 'https://www.baltana.com/files/wallpapers-2/Food-HD-Pictures-04863.jpg'

image_urls = [url_image_1, url_image_2, url_image_3, url_image_4,url_image_5] 

In [19]:
import base64
import requests

def encode_images_to_base64(image_urls):
    """
    Downloads and encodes a list of image URLs to base64 strings.

    Parameters:
    - image_urls (list): A list of image URLs.

    Returns:
    - list: A list of base64-encoded image strings.
    """
    encoded_images = []
    for url in image_urls:
        response = requests.get(url)
        if response.status_code == 200:
            encoded_image = base64.b64encode(response.content).decode("utf-8")
            encoded_images.append(encoded_image)
            print(type(encoded_image))
        else:
            print(f"Warning: Failed to fetch image from {url} (Status code: {response.status_code})")
            encoded_images.append(None)
    return encoded_images

In [20]:
encoded_images = encode_images_to_base64(image_urls)

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>


In [21]:
def generate_model_response(image,user_query):
    response = requests.post("http://localhost:11434/api/generate", json={
        "model": "llava",
        "prompt": user_query,
        "images": [image],
        "stream": False
    })

    data = response.json()
    print(data["response"])

In [ ]:
image = encoded_images[1]

user_query = "How many cars are in this image?"

print("User Query: ", user_query)
generate_model_response(image, user_query)

User Query:  How many cars are in this image?
 There is one car visible in the image, which appears to be a yellow taxi cab parked on the side of the road. 
Model Response:  None


In [ ]:
image = encoded_images[2]

user_query = "How severe is the damage in this image?"

print("User Query: ", user_query)
generate_model_response(image, user_query)

User Query:  How severe is the damage in this image?
 The image shows a significant amount of damage. There's flooding, with water levels reaching up to the rooftops of some houses. This level of water ingress can cause substantial structural damage, including rotting of wood and potential for mold growth inside buildings. The agricultural fields in the background are also likely affected by the floodwaters, which may have damaged crops or rendered the land unusable until it's cleaned up and dried out.

The severity of the damage is high due to the extent of the flooding, with water reaching into areas where it could cause significant harm to property and potentially pose safety risks for the residents if they try to access damaged homes without proper precautions or assistance. 
Model Response:  None


In [ ]:
image = encoded_images[3]

user_query = "How much sodium is in this product?"

print("User Query: ", user_query)
generate_model_response(image, user_query)

User Query:  How much sodium is in this product?
 The image shows a nutrition label for a product with the title "NUTRITION FACTS" and various pieces of information about the contents. However, it appears that the sodium content is not fully visible due to the angle and resolution of the photo.

If we look closely at what is partially visible, there seems to be a percentage icon next to sodium (Na), which typically represents the amount of sodium in relation to the recommended daily intake for an adult person. Unfortunately, the specific number under the sodium symbol is not clearly legible due to the quality of the image.

To provide accurate information about the sodium content, I would need a clearer image or more detailed view of the label. 
Model Response:  None


In [16]:
image = encoded_images[3]
user_query = "How much cholesterol is in this product?"
print("User Query: ", user_query)
generate_model_response(image, user_query)

User Query:  How much cholesterol is in this product?
 The image you've provided appears to be a nutrition label for a product, which includes information about the nutritional content of the item. According to the label, there are several different types of cholesterol listed, each with its corresponding amount per serving:

- Cholesterol: 0 mg
- Sodium: 5 mg
- Total Carbohydrate: 26 g
- Dietary Fiber: 18 g
- Sugar: 9 g
- Protein: 34 g

The nutrition label also shows other nutritional information such as calories, fat content (including saturated and unsaturated fats), vitamins, minerals, and more. However, it is not explicitly stated how much cholesterol is in the product. It is possible that the item contains some form of cholesterol not listed on this label or that the serving size is too small to provide a meaningful amount for consumption. Cholesterol is typically measured in milligrams (mg) and found in various food products, particularly those from animal sources. 


In [17]:
image = encoded_images[1]
user_query = "What is the color of the woman's jacket?"
print("User Query: ", user_query)
print("Model Response: ", generate_model_response(image, user_query))

User Query:  What is the color of the woman's jacket?
 The woman's jacket is yellow. 
Model Response:  None


In [22]:
import base64
import requests
from PIL import Image
import os

image = encoded_images[4]
USDA_API_KEY = os.getenv("USDA_API_KEY") or "UibX72HfLzbSnDqkyBICmaL1RoeERHxgoXlBWadW"


def identify_food_items_llava(image_path):
    response = requests.post("http://localhost:11434/api/generate", json={
        "model": "llava",
        "prompt": "List all recognizable food items in this image. Just list them, comma-separated.",
        "images": [image],
        "stream": False
    })
    return response.json()["response"]

def get_calories_usda(food_name):
    search_url = f"https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {
        "query": food_name,
        "pageSize": 1,
        "api_key": USDA_API_KEY
    }
    response = requests.get(search_url, params=params)
    data = response.json()
    try:
        item = data['foods'][0]
        calories = next((n["value"] for n in item["foodNutrients"] if n["nutrientName"] == "Energy"), None)
        return calories or 0
    except (KeyError, IndexError):
        return 0

def estimate_total_calories(image_path):
    caption = identify_food_items_llava(image_path)
    food_items = [item.strip().lower() for item in caption.split(",")]
    results = []

    for item in food_items:
        cals = get_calories_usda(item)
        results.append({
            "name": item,
            "estimated_calories_per_100g": cals
        })

    total = sum(r["estimated_calories_per_100g"] for r in results)

    return {
        "caption": caption,
        "foods": results,
        "estimated_total_calories": total
    }

# 🧪 Example usage
if __name__ == "__main__":
    image_path = "your_image.jpg"
    result = estimate_total_calories(image_path)
    print(result)


{'caption': ' hamburger, tomato, lettuce, cheese ', 'foods': [{'name': 'hamburger', 'estimated_calories_per_100g': 212}, {'name': 'tomato', 'estimated_calories_per_100g': 0}, {'name': 'lettuce', 'estimated_calories_per_100g': 20}, {'name': 'cheese', 'estimated_calories_per_100g': 331}], 'estimated_total_calories': 563}
